## Seq2seq: преобразование последовательностей (всего 10 баллов)

На этом семинаре мы разберём нейронные сети с архитектурой **энкодер–декодер** и применим их к машинному переводу. **Энкодер** кодирует исходную последовательность, а **декодер** по этому представлению порождает выходную последовательность.

![Схема модели seq2seq](https://esciencegroup.files.wordpress.com/2016/03/seq2seq.jpg)
_Источник изображения: esciencegroup.files.wordpress.com._

Архитектуры энкодер–декодер позволяют преобразовывать данные разных типов. Примеры задач:

* машинный перевод и диалоговые системы;
* [генерация описаний изображений](https://cocodataset.org/#captions-2015) и [image2latex](https://github.com/harvardnlp/im2markup): свёрточный энкодер и рекуррентный декодер;
* [генерация изображений по текстовому описанию](https://arxiv.org/abs/1511.02793): рекуррентный энкодер и свёрточный декодер;
* преобразование графем в фонемы: получение фонетической транскрипции слова по его написанию.

В этой версии семинара сохранены готовые реализации. Разбирайте их по шагам, проверяйте размерности тензоров и сравнивайте поведение моделей без внимания и с вниманием.

## Задача: машинный перевод

Будем переводить описания отелей и хостелов **с русского на английский**. Тематика корпуса достаточно узкая, чтобы обучить учебную модель с умеренными затратами вычислений, но позволяет увидеть основные этапы машинного перевода.

Сначала подготовим данные. В корпусе каждой строке соответствует пара «английский текст — русский текст», разделённая табуляцией. Входом модели будет русский текст, а целевым выходом — английский. Приведём тексты к нижнему регистру и разделим слова и знаки пунктуации с помощью `WordPunctTokenizer`.

При токенизации по словам редкие слова и разные словоформы увеличивают размер словаря. Для русского языка с его богатой морфологией это особенно заметно. Посимвольная токенизация уменьшает словарь, но делает последовательности длиннее. Выберем промежуточный вариант — **субсловную токенизацию**.

Используем [Byte Pair Encoding (BPE)](https://github.com/rsennrich/subword-nmt). Алгоритм начинает с символов и последовательно объединяет наиболее частые пары соседних элементов. После заданного числа объединений частые слова могут стать одним токеном, а редкие слова разбиваются на более короткие фрагменты. Эти фрагменты не обязательно совпадают со слогами или морфемами.

В реализации `subword-nmt` маркер `@@` показывает, что токен продолжается следующим фрагментом: например, `мор@@ ели@@ и` восстанавливается в `морелии`.

**Окружение.** Ноутбук рассчитан на Jupyter или Google Colab. Следующая ячейка устанавливает зависимости и загружает данные и вспомогательный класс `Vocab`; для неё нужен доступ к интернету. Для полного обучения желательно использовать GPU. На CPU сначала выполните короткий пробный запуск, уменьшив число итераций в циклах обучения.

In [ ]:
%pip install "torch>=1.3.0" numpy matplotlib nltk scikit-learn subword-nmt bokeh requests tqdm

from pathlib import Path
import requests

files = {
    'data.txt': 'https://www.dropbox.com/s/yy2zqh34dyhv07i/data.txt?dl=1',
    'vocab.py': 'https://raw.githubusercontent.com/yandexdataschool/nlp_course/2020/week04_seq2seq/vocab.py',
}
for filename, url in files.items():
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    Path(filename).write_bytes(response.content)
    print(f'Загружен файл {filename}: {len(response.content):,} байт')

# Благодарим команды Tilda и DeepHack за данные, Дмитрия Емельяненко — за код.

In [ ]:
from nltk.tokenize import WordPunctTokenizer
from subword_nmt.learn_bpe import learn_bpe
from subword_nmt.apply_bpe import BPE

tokenizer = WordPunctTokenizer()

def tokenize(x):
    return ' '.join(tokenizer.tokenize(x.lower()))

# В корпусе первая колонка английская, вторая — русская.
with open('data.txt', encoding='utf-8') as data_file, \
     open('train.en', 'w', encoding='utf-8') as f_src, \
     open('train.ru', 'w', encoding='utf-8') as f_dst:
    for line_no, line in enumerate(data_file, start=1):
        if not line.strip():
            continue
        parts = line.rstrip('\r\n').split('\t')
        if len(parts) != 2 or not all(part.strip() for part in parts):
            raise ValueError(f'Некорректная пара текстов в строке {line_no}')
        src_line, dst_line = parts
        f_src.write(tokenize(src_line) + '\n')
        f_dst.write(tokenize(dst_line) + '\n')

# Обучаем отдельные правила BPE для каждого языка и применяем их к корпусу.
bpe = {}
for lang in ['en', 'ru']:
    with open('train.' + lang, encoding='utf-8') as f_in, \
         open('bpe_rules.' + lang, 'w', encoding='utf-8') as f_rules:
        learn_bpe(f_in, f_rules, num_symbols=8000)
    with open('bpe_rules.' + lang, encoding='utf-8') as f_rules:
        bpe[lang] = BPE(f_rules)

    with open('train.' + lang, encoding='utf-8') as f_in, \
         open('train.bpe.' + lang, 'w', encoding='utf-8') as f_out:
        for line in f_in:
            f_out.write(bpe[lang].process_line(line.strip()) + '\n')

### Построение словарей

Теперь построим словари, которые сопоставляют токены их числовым идентификаторам и позволяют выполнить обратное преобразование. Они нужны, чтобы подавать тексты в модель и преобразовывать её предсказания в строки.

В классе `Vocab` используются специальные токены:

* **BOS** (`_BOS_`) — начало последовательности;
* **EOS** (`_EOS_`) — конец последовательности;
* **UNK** (`_UNK_`) — токен, отсутствующий в словаре.

Для выравнивания длин последовательностей в батче здесь повторяется EOS: отдельного токена PAD нет. Метод `compute_mask()` включает первый EOS и исключает последующие EOS, которыми заполнен остаток строки.

Отложим 3000 пар для валидации (`dev`). Словари построим только по обучающей части.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# splitlines() не добавляет пустую запись после завершающего перевода строки.
with open('train.bpe.ru', encoding='utf-8') as f_in:
    data_inp = np.array(f_in.read().splitlines())
with open('train.bpe.en', encoding='utf-8') as f_out:
    data_out = np.array(f_out.read().splitlines())
assert len(data_inp) == len(data_out) and len(data_inp) > 3000

from sklearn.model_selection import train_test_split
train_inp, dev_inp, train_out, dev_out = train_test_split(
    data_inp, data_out, test_size=3000, random_state=42
)
for i in range(3):
    print('Вход (ru):', train_inp[i])
    print('Перевод (en):', train_out[i], end='\n\n')

In [ ]:
from vocab import Vocab
inp_voc = Vocab.from_lines(train_inp)
out_voc = Vocab.from_lines(train_out)

In [ ]:
# Преобразуем строки в идентификаторы токенов и обратно.
batch_lines = sorted(train_inp, key=len)[5:10]
batch_ids = inp_voc.to_matrix(batch_lines)
batch_lines_restored = inp_voc.to_lines(batch_ids)

print("Исходные строки")
print(batch_lines)
print("\nТокены → идентификаторы (0 = BOS, 1 = EOS):")
print(batch_ids)
print("\nОбратное преобразование в токены")
print(batch_lines_restored)

Постройте распределения длин исходных текстов и переводов, чтобы оценить размеры последовательностей. Здесь длина измеряется в **BPE-токенах**, без BOS и EOS.

In [ ]:
plt.figure(figsize=[8, 4])
plt.subplot(1, 2, 1)
plt.title("Длина исходного текста")
plt.hist(list(map(len, map(str.split, train_inp))), bins=20);

plt.subplot(1, 2, 2)
plt.title("Длина перевода")
plt.hist(list(map(len, map(str.split, train_out))), bins=20);

### Модель энкодер–декодер без внимания

Ниже приведена базовая модель: однослойный GRU-энкодер и декодер на основе `GRUCell`, без механизма внимания. Она служит отправной точкой для сравнения.

Энкодер обрабатывает входную последовательность. Его состояние на первом EOS преобразуется линейным слоем `dec_start` в начальное состояние декодера. Декодер на каждом шаге получает эмбеддинг предыдущего выходного токена и предсказывает следующий.

При обучении используем **teacher forcing**: подаём декодеру предыдущие токены эталонного перевода. При генерации используем собственные предсказания модели и **жадное декодирование** — выбираем токен с максимальным логитом.

В комментариях к формам тензоров `batch` обозначает размер батча, `time` — число позиций в последовательности, а `hid_size` — размер скрытого состояния.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Фиксируем начальные состояния генераторов случайных чисел.
np.random.seed(42)
torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', device)

In [ ]:
class BasicModel(nn.Module):
    def __init__(self, inp_voc, out_voc, emb_size=64, hid_size=128):
        """
        Простая модель seq2seq с архитектурой энкодер–декодер.
        """
        super().__init__() # Инициализируем базовый класс для регистрации слоёв и параметров.

        self.inp_voc, self.out_voc = inp_voc, out_voc
        self.hid_size = hid_size

        self.emb_inp = nn.Embedding(len(inp_voc), emb_size)
        self.emb_out = nn.Embedding(len(out_voc), emb_size)
        self.enc0 = nn.GRU(emb_size, hid_size, batch_first=True)

        self.dec_start = nn.Linear(hid_size, hid_size)
        self.dec0 = nn.GRUCell(emb_size, hid_size)
        self.logits = nn.Linear(hid_size, len(out_voc))

    def forward(self, inp, out):
        """ Вычисляет логиты для эталонного перевода с teacher forcing. """
        initial_state = self.encode(inp)
        return self.decode(initial_state, out)


    def encode(self, inp, **flags):
        """
        Кодирует входную последовательность и вычисляет начальное состояние декодера.
        :param inp: матрица входных токенов [batch, time]
        :returns: список тензоров начального состояния декодера
        """
        inp_emb = self.emb_inp(inp)
        batch_size = inp.shape[0]

        enc_seq, [last_state_but_not_really] = self.enc0(inp_emb)
        # enc_seq: [batch, time, hid_size], last_state: [batch, hid_size]

        # Из-за заполнения последний шаг GRU не всегда соответствует концу текста.
        lengths = (inp != self.inp_voc.eos_ix).to(torch.int64).sum(dim=1).clamp_max(inp.shape[1] - 1)
        last_state = enc_seq[torch.arange(len(enc_seq), device=inp.device), lengths]
        # Форма: [batch_size, hid_size].

        dec_start = self.dec_start(last_state)
        return [dec_start]

    def decode_step(self, prev_state, prev_tokens, **flags):
        """
        Вычисляет новое состояние и логиты следующего токена по предыдущему состоянию и токену.
        :param prev_state: список тензоров состояния, в формате результата encode(...)
        :param prev_tokens: предыдущие выходные токены, int64[batch_size]
        :return: новое состояние и логиты [batch, len(out_voc)]
        """
        [prev_gru0_state, ] = prev_state

        prev_tokens_embs = self.emb_out(prev_tokens)
        new_dec_state = self.dec0(prev_tokens_embs, prev_gru0_state)
        output_logits = self.logits(new_dec_state)

        return [new_dec_state], output_logits

    def decode(self, initial_state, out_tokens, **flags):
        """ Последовательно вызывает decode_step с токенами эталонного перевода out_tokens. """
        batch_size = out_tokens.shape[0]
        state = initial_state

        # В первой позиции явно задаём предсказание BOS.
        onehot_bos = F.one_hot(torch.full([batch_size], self.out_voc.bos_ix, dtype=torch.int64),
                               num_classes=len(self.out_voc)).to(device=out_tokens.device)
        first_logits = torch.log(onehot_bos.to(torch.float32) + 1e-9)

        logits_sequence = [first_logits]
        for i in range(out_tokens.shape[1] - 1):
            state, logits = self.decode_step(state, out_tokens[:, i])
            logits_sequence.append(logits)
        return torch.stack(logits_sequence, dim=1)

    def decode_inference(self, initial_state, max_len=100, **flags):
        """ Генерирует переводы с помощью жадного декодирования. """
        batch_size, device = len(initial_state[0]), initial_state[0].device
        state = initial_state
        outputs = [torch.full([batch_size], self.out_voc.bos_ix, dtype=torch.int64,
                              device=device)]
        all_states = [initial_state]

        for i in range(max_len):
            state, logits = self.decode_step(state, outputs[-1])
            outputs.append(logits.argmax(dim=-1))
            all_states.append(state)

        return torch.stack(outputs, dim=1), all_states

    @torch.no_grad()
    def translate_lines(self, inp_lines, **kwargs):
        inp = self.inp_voc.to_matrix(inp_lines).to(device)
        initial_state = self.encode(inp)
        out_ids, states = self.decode_inference(initial_state, **kwargs)
        return self.out_voc.to_lines(out_ids.cpu().numpy()), states

In [ ]:
# Проверяем размерности тензоров и работу модели.
model = BasicModel(inp_voc, out_voc).to(device)

dummy_inp_tokens = inp_voc.to_matrix(sorted(train_inp, key=len)[5:10]).to(device)
dummy_out_tokens = out_voc.to_matrix(sorted(train_out, key=len)[5:10]).to(device)

h0 = model.encode(dummy_inp_tokens)
h1, logits1 = model.decode_step(h0, torch.arange(len(dummy_inp_tokens), device=device))

assert isinstance(h1, list) and len(h1) == len(h0)
assert h1[0].shape == h0[0].shape and not torch.allclose(h1[0], h0[0])
assert logits1.shape == (len(dummy_inp_tokens), len(out_voc))

logits_seq = model.decode(h0, dummy_out_tokens)
assert logits_seq.shape == (dummy_out_tokens.shape[0], dummy_out_tokens.shape[1], len(out_voc))

# Проверяем полный прямой проход.
logits_seq2 = model(dummy_inp_tokens, dummy_out_tokens)
assert logits_seq2.shape == logits_seq.shape

In [ ]:
dummy_translations, dummy_states = model.translate_lines(train_inp[:3], max_len=25)
print("Переводы до обучения:")
print('\n'.join([line for line in dummy_translations]))

### Функция потерь (2 балла)

Целевая функция похожа на функцию потерь нейронной языковой модели, но вероятности дополнительно зависят от исходного текста $X$:

$$ L = {\frac1{|D|}} \sum_{(X, Y) \in D} \sum_{y_t \in Y} - \log p(y_t \mid y_1, \dots, y_{t-1}, X, \theta). $$

Здесь $|D|$ — **суммарное число учитываемых токенов в целевых последовательностях**: включая BOS и первый EOS, но исключая заполнение после первого EOS. В этом ноутбуке заполнение выполнено повторными EOS. Маску получаем с помощью `model.out_voc.compute_mask(out)`.

Модель задаёт предсказание BOS в первой позиции явно, поэтому вклад этой позиции в потери практически равен нулю. Остальные позиции рассчитываем через `F.cross_entropy`, затем применяем маску и усредняем по учитываемым токенам. Разберите, как выравниваются предсказания и целевые токены.

In [ ]:
def compute_loss(model, inp, out, **flags):
    """
    Вычисляет средние потери по учитываемым целевым токенам, скаляр float32.
    :param inp: входные токены, int64[batch, time]
    :param out: токены эталонного перевода, int64[batch, time]

    Учитываем BOS и первый EOS; заполнение после первого EOS исключаем.
    Нормируем сумму потерь на число целевых токенов, для которых маска истинна.
    """
    mask = model.out_voc.compute_mask(out)  # [batch_size, out_len]
    logits_seq = model(inp, out)  # [batch_size, out_len, num_tokens]
    num_tokens = logits_seq.shape[-1]

    # Не создаём большой one-hot-тензор: cross_entropy принимает индексы токенов.
    per_token_loss = F.cross_entropy(
        logits_seq.reshape(-1, num_tokens), out.reshape(-1), reduction='none'
    )
    return (per_token_loss * mask.reshape(-1)).sum() / mask.sum()

In [ ]:
dummy_loss = compute_loss(model, dummy_inp_tokens, dummy_out_tokens)
print("Потери:", dummy_loss)
assert torch.isfinite(dummy_loss).item() and dummy_loss.item() > 0, "Потери должны быть конечными и положительными"

# Проверяем обратное распространение ошибки.
dummy_loss.backward()
for name, param in model.named_parameters():
    assert param.grad is not None and param.grad.abs().max().item() > 0, f"Параметр {name} не получил ненулевых градиентов"

### Оценка качества: BLEU

Для оценки машинного перевода используем [BLEU](https://en.wikipedia.org/wiki/BLEU). Метрика сравнивает n-граммы перевода с n-граммами эталона для $n = 1, 2, 3, 4$. Число совпадений ограничивается числом вхождений соответствующей n-граммы в эталоне. Затем берётся геометрическое среднее точностей и применяется штраф за слишком короткий перевод.

BLEU оценивает совпадение формулировок с эталоном и [имеет ограничения](http://www.cs.jhu.edu/~ccb/publications/re-evaluating-the-role-of-bleu-in-mt-research.pdf). Поэтому наряду с числом BLEU полезно читать сами переводы.

В этом семинаре вычисляем **корпусный BLEU-4 без сглаживания**, на шкале от 0 до 100. Перед оценкой убираем маркеры BPE. Переводы генерируем небольшими батчами, но BLEU считаем один раз по всему корпусу: среднее BLEU отдельных батчей было бы другой метрикой.

До обучения BLEU может быть близок к нулю. Предупреждения NLTK об отсутствии совпадающих n-грамм в таком запуске ожидаемы.

Токенизация и настройки метрики влияют на результат. Для сопоставимых опубликованных результатов используйте [SacreBLEU](https://pypi.org/project/sacrebleu/) и указывайте конфигурацию оценки.

In [ ]:
from nltk.translate.bleu_score import corpus_bleu

def compute_bleu(model, inp_lines, out_lines, bpe_sep='@@ ', batch_size=32, **flags):
    """
    Вычисляет корпусный BLEU-4 без сглаживания, на шкале от 0 до 100.
    Генерирует переводы батчами; метрику рассчитывает по всему корпусу.
    Для публикации сопоставимых результатов используйте SacreBLEU.
    """
    if len(inp_lines) != len(out_lines) or len(inp_lines) == 0:
        raise ValueError('Нужны непустые списки входов и эталонов одинаковой длины')
    if batch_size <= 0:
        raise ValueError('Размер батча должен быть положительным')

    flags = dict(flags)
    flags['return_attention'] = False
    translations = []
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            for start in range(0, len(inp_lines), batch_size):
                batch = inp_lines[start:start + batch_size]
                # У моделей разные дополнительные результаты; переводы идут первыми.
                translations.extend(model.translate_lines(batch, **flags)[0])
    finally:
        model.train(was_training)

    translations = [line.replace(bpe_sep, '') for line in translations]
    actual = [line.replace(bpe_sep, '') for line in out_lines]
    return corpus_bleu(
        [[ref.split()] for ref in actual],
        [translation.split() for translation in translations],
    ) * 100

In [ ]:
compute_bleu(model, dev_inp, dev_out)

### Цикл обучения

Обучение модели энкодер–декодер состоит из знакомых шагов: случайно выбираем батч, вычисляем потери, обнуляем градиенты, выполняем обратное распространение ошибки и обновляем параметры.

Периодически оцениваем BLEU на валидационной части и строим графики. Оценка выполняется без градиентов; после неё модель возвращается в прежний режим.

В полном запуске предусмотрено 25 000 итераций. Для проверки окружения начните с нескольких итераций; при таком запуске качество перевода ещё не показательно.

In [ ]:
from IPython.display import clear_output
from tqdm import tqdm, trange
metrics = {'train_loss': [], 'dev_bleu': [] }

model = BasicModel(inp_voc, out_voc).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
batch_size = 32

In [ ]:
for _ in trange(25000):
    step = len(metrics['train_loss']) + 1
    batch_ix = np.random.randint(len(train_inp), size=batch_size)
    batch_inp = inp_voc.to_matrix(train_inp[batch_ix]).to(device)
    batch_out = out_voc.to_matrix(train_out[batch_ix]).to(device)

    # Шаг обучения на batch_inp и batch_out.
    loss_t = compute_loss(model, batch_inp, batch_out)
    opt.zero_grad()
    loss_t.backward()
    opt.step()

    metrics['train_loss'].append((step, loss_t.item()))

    if step % 100 == 0:
        metrics['dev_bleu'].append((step, compute_bleu(model, dev_inp, dev_out)))

        clear_output(True)
        plt.figure(figsize=(12,4))
        for i, (name, history) in enumerate(sorted(metrics.items())):
            plt.subplot(1, len(metrics), i + 1)
            plt.title({'train_loss': 'Потери на обучении', 'dev_bleu': 'BLEU на валидации'}[name])
            plt.plot(*zip(*history))
            plt.grid()
        plt.show()
        print("Средние потери = %.3f" % np.mean(metrics['train_loss'][-10:], axis=0)[1], flush=True)

# BLEU может колебаться; оценивайте общую динамику на большом числе итераций.

In [ ]:
# BLEU — показатель для сравнения моделей, а не проверка корректности кода.
# Сравнивайте результаты при одинаковой конфигурации метрики.
basic_bleu = compute_bleu(model, dev_inp, dev_out)
print('BLEU базовой модели: %.2f' % basic_bleu)

In [ ]:
for inp_line, trans_line in zip(dev_inp[::500], model.translate_lines(dev_inp[::500])[0]):
    print(inp_line)
    print(trans_line)
    print()

### Добавляем механизм внимания

Теперь расширим базовую модель с помощью **механизма внимания** (*attention*). Вместо единственного сжатого представления входа декодер сможет на каждом шаге обращаться к разным позициям исходной последовательности.

Разберём два компонента: **слой внимания** и **модель seq2seq с вниманием**. Затем сравним модели по BLEU и рассмотрим карты внимания.

### Слой внимания (3 балла)

Рассмотрим **аддитивное внимание**. Пусть даны состояния энкодера $h^e_0, h^e_1, \dots, h^e_T$ и предыдущее состояние декодера $h^d$.

1. Вычислим оценку соответствия каждой позиции входа состоянию декодера с помощью нейронной сети:

   $$ a_t = \operatorname{linear}_{out}\!\left(\tanh\!\left(\operatorname{linear}_{e}(h^e_t) + \operatorname{linear}_{d}(h^d)\right)\right). $$

2. До применения softmax замаскируем позиции, которые не должны участвовать во внимании. В приведённой модели исключаются первый EOS и всё заполнение после него; BOS остаётся доступен.

3. Преобразуем оценки допустимых позиций в веса с помощью softmax:

   $$ p_t = \frac{e^{a_t}}{\sum_{\tau} e^{a_{\tau}}}. $$

4. Получим **вектор контекста** — взвешенную сумму состояний энкодера:

   $$ \operatorname{attn} = \sum_t p_t h^e_t. $$

Обратите внимание: промежуточные линейные преобразования имеют размер `hid_size`, а вектор контекста — `enc_size`.

Подробнее о внимании можно прочитать в слайдах лекции и [статье Distill](https://distill.pub/2016/augmented-rnns/).

In [ ]:
class AttentionLayer(nn.Module):
    def __init__(self, name, enc_size, dec_size, hid_size, activ=torch.tanh):
        """ Слой аддитивного внимания: вычисляет вектор контекста и веса. """
        super().__init__()
        self.name = name
        self.enc_size = enc_size # Размер состояния энкодера.
        self.dec_size = dec_size # Размер состояния декодера.
        self.hid_size = hid_size # Размер скрытого слоя внимания.
        self.activ = activ       # Нелинейность скрытого слоя внимания.

        # Создаём обучаемые линейные преобразования.
        self.linear_d = nn.Linear(dec_size, hid_size)
        self.linear_e = nn.Linear(enc_size, hid_size)
        self.linear_out = nn.Linear(hid_size, 1)


    def forward(self, enc, dec, inp_mask):
        """
        Вычисляет вектор контекста и веса внимания.
        :param enc: состояния энкодера, float32[batch_size, ninp, enc_size]
        :param dec: состояние декодера — запрос, float32[batch_size, dec_size]
        :param inp_mask: маска входа [batch_size, ninp]; 0 для EOS и заполнения
        :returns: attn[batch_size, enc_size], probs[batch_size, ninp]
            - attn: вектор контекста, взвешенная сумма enc, [batch_size, enc_size]
            - probs: веса внимания после softmax, [batch_size, ninp]
        """

        # Вычисляем оценки соответствия.
        dec_transformed = self.linear_d(dec).unsqueeze(1) # [batch_size, 1, hid_size]
        enc_transformed = self.linear_e(enc) # [batch_size, ninp, hid_size]
        combined = self.activ(enc_transformed + dec_transformed) # [batch_size, ninp, hid_size]
        logits = self.linear_out(combined).squeeze(-1) # [batch_size, ninp]

        # Для исключённых позиций задаём большое отрицательное значение до softmax.
        # Используем torch.where для применения маски.

        masked_logits = torch.where(inp_mask > 0, logits, torch.full_like(logits, -1e9)) # [batch_size, ninp]

        # Получаем веса внимания с помощью softmax.
        probs = F.softmax(masked_logits, dim=1) # [batch_size, ninp]

        # Вычисляем взвешенную сумму состояний энкодера.
        attn = torch.sum(probs.unsqueeze(-1) * enc, dim=1) # [batch_size, enc_size]

        return attn, probs

### Модель seq2seq с вниманием (3 балла)

Используем слой внимания внутри декодера:

![Использование внимания в рекуррентной модели](https://i.imgur.com/6fKHlHb.png)
_Источник: [статья Distill](https://distill.pub/2016/augmented-rnns/)._

На каждом шаге **предыдущее состояние декодера** выступает запросом к состояниям энкодера. Полученный вектор контекста объединяем с эмбеддингом предыдущего выходного токена и передаём в `GRUCell` декодера. Поэтому размер входа этой ячейки равен `emb_size + hid_size`.

Ключевая деталь реализации — **состояние модели**. Метод `encode()` возвращает список тензоров, которые затем доступны в каждом вызове `decode_step()`. В этой модели список содержит:

* скрытое состояние декодера;
* всю последовательность состояний энкодера;
* маску входной последовательности.

Кроме нового состояния и логитов, `decode_step()` возвращает веса внимания. При генерации они собираются в массив формы `[batch_size, translation_length, input_length]`; его первая строка по оси времени соответствует первому предсказанному токену после BOS. Параметр `return_attention=False` позволяет не сохранять веса при расчёте BLEU.

Есть и другие способы включить внимание в модель. Для дополнительных экспериментов можно обратиться к [этой](https://arxiv.org/abs/1609.08144), [этой](https://arxiv.org/abs/1706.03762) и [этой](https://arxiv.org/abs/1808.03867) работам. Для генерации описаний изображений и распознавания формул применяют [визуальное внимание](https://arxiv.org/abs/1502.03044).

In [ ]:
class AttentiveModel_1(nn.Module):
    def __init__(self, inp_voc, out_voc, emb_size=64, hid_size=128, attn_size=128):
        """
        Простая модель seq2seq с архитектурой энкодер–декодер и вниманием.
        """
        super().__init__()  # Инициализируем базовый класс для регистрации слоёв и параметров.

        self.inp_voc, self.out_voc = inp_voc, out_voc
        self.hid_size = hid_size

        self.emb_inp = nn.Embedding(len(inp_voc), emb_size)
        self.emb_out = nn.Embedding(len(out_voc), emb_size)
        self.enc0 = nn.GRU(emb_size, hid_size, batch_first=True)

        self.attention = AttentionLayer(
            name='attention',
            enc_size=hid_size,
            dec_size=hid_size,
            hid_size=attn_size
        )

        self.dec_start = nn.Linear(hid_size, hid_size)
        self.dec0 = nn.GRUCell(emb_size + hid_size, hid_size)  # На вход подаём также вектор контекста внимания.
        self.logits = nn.Linear(hid_size, len(out_voc))

    def forward(self, inp, out):
        """ Вычисляет логиты для эталонного перевода с teacher forcing. """
        initial_state = self.encode(inp)
        return self.decode(initial_state, out)

    def compute_mask(self, inp):
        inp_mask = (inp != self.inp_voc.eos_ix).to(torch.int64)
        return inp_mask

    def encode(self, inp, **flags):
        """
        Кодирует входную последовательность и вычисляет начальное состояние декодера.
        :param inp: матрица входных токенов [batch, time]
        :returns: список тензоров начального состояния декодера
        """
        inp_emb = self.emb_inp(inp)
        enc_seq, _ = self.enc0(inp_emb)  # [batch, time, hid_size]

        # Строим маску позиций, доступных механизму внимания.
        inp_mask = self.compute_mask(inp)

        # Берём состояние энкодера на первом EOS.
        lengths = inp_mask.sum(dim=1).clamp_max(inp.shape[1] - 1)
        last_state = enc_seq[torch.arange(len(enc_seq), device=inp.device), lengths]  # [batch_size, hid_size]

        dec_start = self.dec_start(last_state)
        return [dec_start, enc_seq, inp_mask]  # Сохраняем также последовательность энкодера и маску.

    def decode_step(self, prev_state, prev_tokens, **flags):
        """
        Вычисляет новое состояние и логиты следующего токена по предыдущему состоянию и токену.
        :param prev_state: список тензоров предыдущего состояния декодера
        :param prev_tokens: предыдущие выходные токены, int64[batch_size]
        :return: новое состояние, логиты [batch, len(out_voc)] и веса внимания
        """
        [prev_gru0_state, enc_seq, inp_mask] = prev_state

        prev_tokens_embs = self.emb_out(prev_tokens)

        # Вычисляем контекст и веса внимания.
        attn_state, attn_probs = self.attention(enc_seq, prev_gru0_state, inp_mask)

        # Объединяем контекст с эмбеддингом предыдущего выходного токена.
        input_to_decoder = torch.cat((prev_tokens_embs, attn_state), dim=-1)  # [batch_size, emb_size + hid_size]
        new_dec_state = self.dec0(input_to_decoder, prev_gru0_state)
        output_logits = self.logits(new_dec_state)

        new_state = [new_dec_state, enc_seq, inp_mask]
        return new_state, output_logits, attn_probs  # Возвращаем новое состояние, логиты и веса внимания.

    def decode(self, initial_state, out_tokens, **flags):
        """ Последовательно вызывает decode_step с токенами эталонного перевода out_tokens. """
        batch_size = out_tokens.shape[0]
        state = initial_state

        # В первой позиции явно задаём предсказание BOS.
        onehot_bos = F.one_hot(torch.full([batch_size], self.out_voc.bos_ix, dtype=torch.int64),
                               num_classes=len(self.out_voc)).to(device=out_tokens.device)
        first_logits = torch.log(onehot_bos.to(torch.float32) + 1e-9)

        logits_sequence = [first_logits]
        for i in range(out_tokens.shape[1] - 1):
            state, logits, _ = self.decode_step(state, out_tokens[:, i])
            logits_sequence.append(logits)
        return torch.stack(logits_sequence, dim=1)

    def decode_inference(self, initial_state, max_len=100, return_attention=True, **flags):
        """ Генерирует переводы с помощью жадного декодирования. """
        batch_size, device = len(initial_state[0]), initial_state[0].device
        state = initial_state
        outputs = [torch.full([batch_size], self.out_voc.bos_ix, dtype=torch.int64,
                              device=device)]
        all_states = [initial_state]
        all_attn_probs = []  # Сохраняем веса внимания каждого шага.

        for i in range(max_len):
            state, logits, attn_probs = self.decode_step(state, outputs[-1])
            outputs.append(logits.argmax(dim=-1))
            all_states.append(state)
            if return_attention:
                all_attn_probs.append(attn_probs.detach().cpu().numpy())

        # Каждый шаг имеет форму [batch, input_length]; собираем шаги по оси 1.
        attention_probs = np.stack(all_attn_probs, axis=1) if all_attn_probs else None
        return torch.stack(outputs, dim=1), all_states, attention_probs

    @torch.no_grad()
    def translate_lines(self, inp_lines, **kwargs):
        inp = self.inp_voc.to_matrix(inp_lines).to(device)
        initial_state = self.encode(inp)
        out_ids, states, attn_probs = self.decode_inference(initial_state, **kwargs)
        translations = self.out_voc.to_lines(out_ids.cpu().numpy())
        return translations, states, attn_probs  # Возвращаем также веса внимания.

In [ ]:
# Проверяем размерности тензоров и работу модели.
model = AttentiveModel_1(inp_voc, out_voc).to(device)

dummy_inp_tokens = inp_voc.to_matrix(sorted(train_inp, key=len)[5:10]).to(device)
dummy_out_tokens = out_voc.to_matrix(sorted(train_out, key=len)[5:10]).to(device)

h0 = model.encode(dummy_inp_tokens)
h1, logits1, attn_probs = model.decode_step(h0, torch.arange(len(dummy_inp_tokens), device=device))

print([e.shape for e in h0])
print([e.shape for e in h1])

assert isinstance(h1, list)
assert len(h1) == len(h0)
assert h1[0].shape == h0[0].shape and not torch.allclose(h1[0], h0[0])
assert logits1.shape == (len(dummy_inp_tokens), len(out_voc))

logits_seq = model.decode(h0, dummy_out_tokens)
assert logits_seq.shape == (dummy_out_tokens.shape[0], dummy_out_tokens.shape[1], len(out_voc))

# Проверяем полный прямой проход.
logits_seq2 = model(dummy_inp_tokens, dummy_out_tokens)
assert logits_seq2.shape == logits_seq.shape

In [ ]:
dummy_translations, dummy_states, dummy_attn_probs = model.translate_lines(train_inp[:3], max_len=25)
print("Переводы до обучения:")
print('\n'.join([line for line in dummy_translations]))

In [ ]:
dummy_loss = compute_loss(model, dummy_inp_tokens, dummy_out_tokens)
print("Потери:", dummy_loss)
assert torch.isfinite(dummy_loss).item() and dummy_loss.item() > 0, "Потери должны быть конечными и положительными"

# Проверяем обратное распространение ошибки.
dummy_loss.backward()
for name, param in model.named_parameters():
    assert param.grad is not None and param.grad.abs().max().item() > 0, f"Параметр {name} не получил ненулевых градиентов"

### Обучение модели с вниманием

Повторно используем функцию потерь, оценку BLEU и цикл обучения. Сравните качество и примеры переводов с базовой моделью.

Обратите внимание: ниже размер батча равен 128, а для базовой модели он был равен 32. Это влияет на число обработанных примеров и время обучения. Чтобы сравнить именно влияние внимания, проведите дополнительный эксперимент с одинаковыми размером батча и числом обновлений.

In [ ]:
# Создаём модель с вниманием, оптимизатор и историю метрик.
from IPython.display import clear_output
from tqdm import tqdm, trange
metrics = {'train_loss': [], 'dev_bleu': [] }

model = AttentiveModel_1(inp_voc, out_voc).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
batch_size = 128

In [ ]:
# Цикл обучения.
for _ in trange(25000):
    step = len(metrics['train_loss']) + 1
    batch_ix = np.random.randint(len(train_inp), size=batch_size)
    batch_inp = inp_voc.to_matrix(train_inp[batch_ix]).to(device)
    batch_out = out_voc.to_matrix(train_out[batch_ix]).to(device)

    # Шаг обучения на batch_inp и batch_out.
    loss_t = compute_loss(model, batch_inp, batch_out)
    opt.zero_grad()
    loss_t.backward()
    opt.step()

    metrics['train_loss'].append((step, loss_t.item()))

    if step % 500 == 0:
        metrics['dev_bleu'].append((step, compute_bleu(model, dev_inp, dev_out)))

        clear_output(True)
        plt.figure(figsize=(12,4))
        for i, (name, history) in enumerate(sorted(metrics.items())):
            plt.subplot(1, len(metrics), i + 1)
            plt.title({'train_loss': 'Потери на обучении', 'dev_bleu': 'BLEU на валидации'}[name])
            plt.plot(*zip(*history))
            plt.grid()
        plt.show()
        print("Средние потери = %.3f" % np.mean(metrics['train_loss'][-10:], axis=0)[1], flush=True)

# BLEU может колебаться; оценивайте общую динамику на большом числе итераций.

In [ ]:
attentive_bleu = compute_bleu(model, dev_inp, dev_out)
print('BLEU модели с вниманием: %.2f' % attentive_bleu)

# Рассматриваем примеры перевода после обучения.
for inp_line, trans_line in zip(dev_inp[::500], model.translate_lines(dev_inp[::500])[0]):
    print('Вход (ru):', inp_line)
    print('Перевод модели (en):', trans_line)
    print()

### Визуализация внимания модели (2 балла)

После обучения посмотрим, какие позиции исходного текста получают большой вес при генерации каждого токена перевода.

Функция ниже строит карты внимания с помощью [Bokeh](https://docs.bokeh.org/en/latest/). По горизонтали расположены входные токены, по вертикали — сгенерированные токены перевода. BOS не включается в выходные подписи, поскольку отдельного шага внимания для него нет. EOS и последующие шаги генерации после конца перевода также не показываются.

Рассмотрите не менее трёх примеров. Найдите понятные соответствия между русскими и английскими фрагментами, случаи изменения порядка слов и ошибки перевода. Не каждая карта обязательно будет близка к диагонали: порядок слов в языках различается.

Для сохранения рисунка можно использовать кнопку сохранения на панели Bokeh. При желании используйте другой способ визуализации.

In [ ]:
import bokeh.plotting as pl
import bokeh.models as bm
from bokeh.io import output_notebook, show
output_notebook()

def draw_attention(inp_line, translation, probs):
    """Рисует веса внимания: столбцы — входные токены, строки — токены перевода."""
    inp_tokens = inp_voc.tokenize(inp_line)
    # Первая строка probs относится к первому предсказанию после BOS.
    # Отображаем только токены текста перевода, без добавленных BOS и EOS.
    trans_tokens = out_voc.tokenize(translation)[1:-1]
    if not trans_tokens:
        print('Перевод пуст: карту внимания для текстовых токенов не строим.')
        return None
    probs = np.asarray(probs)
    if probs.ndim != 2 or probs.shape[0] < len(trans_tokens) or probs.shape[1] < len(inp_tokens):
        raise ValueError('Ожидается матрица [шаг перевода, позиция входа] достаточного размера')
    probs = probs[:len(trans_tokens), :len(inp_tokens)]

    fig = pl.figure(
        x_range=(0, len(inp_tokens)), y_range=(0, len(trans_tokens)),
        x_axis_type=None, y_axis_type=None, tools='save,reset',
        width=800, height=max(300, min(800, 25 * len(trans_tokens))),
        title='Веса внимания',
    )
    fig.image([probs[::-1]], 0, 0, len(inp_tokens), len(trans_tokens))

    fig.add_layout(bm.LinearAxis(axis_label='Токены исходного текста'), 'above')
    fig.xaxis.ticker = np.arange(len(inp_tokens)) + 0.5
    fig.xaxis.major_label_overrides = dict(zip(np.arange(len(inp_tokens)) + 0.5, inp_tokens))
    fig.xaxis.major_label_orientation = np.pi / 4  # Bokeh принимает угол в радианах.

    fig.add_layout(bm.LinearAxis(axis_label='Токены перевода'), 'left')
    fig.yaxis.ticker = np.arange(len(trans_tokens)) + 0.5
    fig.yaxis.major_label_overrides = dict(zip(np.arange(len(trans_tokens)) + 0.5, trans_tokens[::-1]))

    show(fig)
    return fig

In [ ]:
inp = dev_inp[::500]

trans, states, attention_probs = model.translate_lines(inp, return_attention=True)

# Веса уже имеют форму [batch_size, translation_length, input_length].
# В этой реализации они возвращаются отдельно от списка состояний.
assert attention_probs.ndim == 3 and attention_probs.shape[0] == len(inp)

In [ ]:
for i in range(min(5, len(inp))):
    draw_attention(inp[i], trans[i], attention_probs[i])

# Рассмотрите соответствия токенов и сопоставьте карты с качеством перевода.

**Если карты внимания трудно интерпретировать**, попробуйте в качестве эксперимента инициализировать **декодер** нулевым состоянием вместо `dec_start`. Так вы уберёте прямой путь передачи информации через начальное состояние декодера; информация об исходном тексте будет поступать через внимание. Сравните качество и карты до и после изменения.

Веса внимания помогают исследовать модель, но сами по себе не гарантируют правильного перевода.

## Дополнительные эксперименты (от 2 дополнительных баллов за эксперимент)

Попробуйте улучшить модель и объяснить, что повлияло на результат. Возможные направления:

* другие рекуррентные ячейки: RNN, GRU, LSTM; более глубокие архитектуры;
* двунаправленный энкодер и другие варианты внимания: аддитивное, скалярное произведение, многоголовое;
* случайное удаление или замена входных токенов (*word dropout*), изменение темпа обучения и другие способы регуляризации;
* замена жадного декодирования на **лучевой поиск** (*beam search*).

Для лучшего понимания seq2seq проведите хотя бы один эксперимент из списка или предложите свой. В коротком отчёте укажите гипотезу, изменения, настройки обучения, полученный BLEU и несколько примеров переводов. Для сравнения используйте одинаковую валидационную выборку и одну конфигурацию метрики.

_Здесь можно записать гипотезу, настройки эксперимента, результаты и выводы._